# PS-13 — Review Authenticity & Counterfeit Detection Agent

This notebook implements a Colab-first autonomous marketplace trust investigation system that operates on `product_url` and `seller_id`, collects public evidence, runs deterministic risk analysis, and outputs a human-investigator-ready report without automatically deciding guilt.

## 01. Project Overview

The system follows the required architecture: live collection -> review, product, and seller pipelines -> evidence fusion -> policy mapping -> deterministic trust risk score -> final report. It is designed for public evidence only, with a clear distinction between observed evidence, inferred risk, and confirmed facts.

In [1]:

# @title 01. Install Dependencies
!pip install -q --upgrade pip
!pip install -q openai tavily-python firecrawl-py playwright sentence-transformers faiss-cpu streamlit
!pip install -q networkx pandas numpy scipy scikit-learn requests beautifulsoup4 lxml
!pip install -q python-dotenv open-clip-torch pillow opencv-python-headless
!playwright install --with-deps chromium


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 14.9 MB/s eta 0:00:00
Installing dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease
Get:4 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.3 MB]
Get:10 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,545 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [52.1 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-updates/unive

## 02. API Key Configuration

Keep all secrets in environment variables or Colab Secrets. Never hardcode keys or print them.

In [2]:
# @title 02. API Key Configuration
import os
import logging

# In Colab, open the Secrets panel, add GROQ_API_KEY, and enable notebook access.
# TAVILY_API_KEY and FIRECRAWL_API_KEY are optional collection services.
try:
    from google.colab import userdata
except ImportError:
    userdata = None

secret_names = ['GROQ_API_KEY', 'TAVILY_API_KEY', 'FIRECRAWL_API_KEY']
for name in secret_names:
    value = os.getenv(name)
    if not value and userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if value:
        os.environ[name] = value

if not os.getenv('GROQ_API_KEY'):
    raise RuntimeError(
        'Groq API key is required. In Colab, open the Secrets panel, add '
        'GROQ_API_KEY with your Groq key, and enable notebook access. Do not paste '
        'the key into notebook code or share it in chat.'
    )

if not os.getenv('TAVILY_API_KEY'):
    logging.warning('TAVILY_API_KEY is not set; web search, seller research, and claim verification will be unavailable.')
if not os.getenv('FIRECRAWL_API_KEY'):
    logging.warning('FIRECRAWL_API_KEY is not set; page collection will try the Playwright fallback.')

GROQ_API_KEY = os.environ['GROQ_API_KEY']
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')
FIRECRAWL_API_KEY = os.getenv('FIRECRAWL_API_KEY')
print('Groq key loaded; optional services are configured when their keys are available.')


Groq key loaded; optional services are configured when their keys are available.


## 03. Imports and Configuration

In [3]:

# @title 03. Imports and Configuration
import json
import os
import re
import logging
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional
from dataclasses import dataclass, field
from urllib.parse import urlparse

import numpy as np
import pandas as pd
from openai import OpenAI
from tavily import TavilyClient
from firecrawl import Firecrawl
from playwright.sync_api import sync_playwright
from sklearn.ensemble import IsolationForest
from sentence_transformers import SentenceTransformer
import faiss
import networkx as nx
import torch
import open_clip
from PIL import Image, ImageFile
from io import BytesIO
import requests

ImageFile.LOAD_TRUNCATED_IMAGES = True
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger('ps13')

MODEL = 'openai/gpt-oss-120b'
RISK_LEVELS = [('CRITICAL', 81, 100), ('HIGH', 61, 80), ('MEDIUM', 31, 60), ('LOW', 0, 30)]
WEIGHTS = {'statistical': 0.15, 'linguistic': 0.15, 'reviewer_network': 0.20, 'claim_risk': 0.15, 'visual': 0.15, 'seller_risk': 0.20}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9

client = OpenAI(api_key=os.environ['GROQ_API_KEY'], base_url='https://api.groq.com/openai/v1')

try:
    tavily_client = TavilyClient(api_key=os.environ['TAVILY_API_KEY'])
except Exception as exc:
    tavily_client = None
    logger.warning('Tavily unavailable: %s', exc)

try:
    firecrawl_client = Firecrawl(api_key=os.environ['FIRECRAWL_API_KEY'])
except Exception as exc:
    firecrawl_client = None
    logger.warning('Firecrawl unavailable: %s', exc)

try:
    sentence_model = SentenceTransformer('all-MiniLM-L6-v2')
except Exception as exc:
    sentence_model = None
    logger.warning('SentenceTransformer unavailable: %s', exc)

try:
    clip_model, _, clip_preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
    clip_model.eval()
    clip_device = 'cuda' if torch.cuda.is_available() else 'cpu'
    clip_model.to(clip_device)
except Exception as exc:
    clip_model = None
    clip_preprocess = None
    logger.warning('CLIP unavailable: %s', exc)

@dataclass
class InvestigationState:
    evidence: List[Dict[str, Any]] = field(default_factory=list)
    product_data: Dict[str, Any] = field(default_factory=dict)
    review_data: Dict[str, Any] = field(default_factory=dict)

state = InvestigationState()

def now_iso():
    return datetime.now(timezone.utc).isoformat()

def safe_float(value, default=0.0):
    try:
        return float(value)
    except (TypeError, ValueError):
        return default

def clamp01(value):
    return max(0.0, min(1.0, float(value)))

def evidence_id(prefix='E'):
    return f'{prefix}{len(state.evidence) + 1:03d}'


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (6 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


## 04. Groq Tool Calling and Orchestration

In [4]:

# @title 04. Groq Tool Calling and Orchestration
def groq_chat(messages, tools=None, temperature=0.1):
    kwargs = {'model': MODEL, 'messages': messages, 'temperature': temperature}
    if tools:
        kwargs['tools'] = tools
        kwargs['tool_choice'] = 'auto'
    return client.chat.completions.create(**kwargs)

def groq_extract_json(prompt, schema_name='object', fallback=None):
    messages = [
        {'role': 'system', 'content': f'You are a careful compliance analyst. Return only valid JSON for the {schema_name}. Never invent missing data.'},
        {'role': 'user', 'content': prompt},
    ]
    try:
        text = groq_chat(messages).choices[0].message.content
        return json.loads(text) if text else (fallback or {})
    except Exception as exc:
        logger.warning('Groq JSON extraction failed: %s', exc)
        return fallback or {}

def orchestrator_decide(product_url, seller_id):
    prompt = f"""
    You are the marketplace trust orchestrator.
    Decide which public collection actions are needed for: {product_url}, seller_id={seller_id}.
    Allowed tools: ['collect_product_page', 'collect_reviews', 'search_web', 'collect_seller_information', 'find_related_listings'].
    Return a JSON array of tool names, no hallucinated evidence.
    """
    result = groq_extract_json(prompt, 'tool plan', fallback={'tools': []})
    tools = result.get('tools') or result.get('tool_names') or []
    return [tool for tool in tools if tool in ['collect_product_page', 'collect_reviews', 'search_web', 'collect_seller_information', 'find_related_listings']]


## 05. Web Search and Page Collection

In [5]:
# @title 05. Web Search and Page Collection
def search_web(query, max_results=5):
    if tavily_client is None:
        return {'results': [], 'status': 'unavailable', 'message': 'Tavily unavailable'}
    try:
        resp = tavily_client.search(query=query, max_results=max_results, search_depth='basic')
        return {'results': resp.get('results', []), 'status': 'ok'}
    except Exception as exc:
        logger.warning('Tavily failed: %s', exc)
        return {'results': [], 'status': 'error', 'message': str(exc)}

def _firecrawl_scrape(url):
    """Call the current Firecrawl Python SDK and normalize its response."""
    result = firecrawl_client.scrape(url, formats=['markdown', 'html'])
    if isinstance(result, dict):
        return result
    if hasattr(result, 'model_dump'):
        return result.model_dump()
    return {
        'markdown': getattr(result, 'markdown', None),
        'html': getattr(result, 'html', None),
        'metadata': getattr(result, 'metadata', None),
    }

def _collect_page_content(url):
    firecrawl_error = None
    if firecrawl_client is not None:
        try:
            return _firecrawl_scrape(url), 'firecrawl', None
        except Exception as exc:
            firecrawl_error = str(exc)
            logger.warning('Firecrawl scrape failed; trying Playwright fallback: %s', exc)

    try:
        with sync_playwright() as p:
            browser = p.chromium.launch(headless=True)
            try:
                page = browser.new_page()
                page.goto(url, wait_until='domcontentloaded', timeout=60000)
                page.wait_for_timeout(1500)
                html = page.content()
            finally:
                browser.close()
        return {'html': html}, 'playwright', firecrawl_error
    except Exception as exc:
        logger.warning('Playwright page collection failed: %s', exc)
        raise RuntimeError(
            f'Public page collection failed. Firecrawl error: {firecrawl_error or "unavailable"}; '
            f'Playwright error: {exc}'
        ) from exc

def collect_product_page(url):
    payload = {'url': url, 'fetched_at': now_iso(), 'status': 'unavailable'}
    try:
        data, method, firecrawl_error = _collect_page_content(url)
        payload = {'url': url, 'fetched_at': now_iso(), 'status': 'ok', 'method': method, 'data': data}
        if firecrawl_error:
            payload['fallback_note'] = f'Firecrawl failed; Playwright succeeded: {firecrawl_error}'
    except Exception as exc:
        payload.update({'status': 'error', 'message': str(exc)})
    return payload

def collect_reviews(url):
    result = {'reviews': [], 'coverage_notes': [], 'source_url': url, 'retrieved_at': now_iso(), 'status': 'unavailable'}
    try:
        data, method, firecrawl_error = _collect_page_content(url)
        raw_content = data.get('markdown') or data.get('html')
        if not raw_content:
            raise RuntimeError('Page collector returned no Markdown or HTML content.')
        result.update({'status': 'ok', 'raw_content': raw_content, 'extraction_method': method})
        result['coverage_notes'].append(f'Collected public page content using {method}. This does not guarantee that reviews were present.')
        if firecrawl_error:
            result['coverage_notes'].append(f'Firecrawl failed and Playwright fallback was used: {firecrawl_error}')
    except Exception as exc:
        logger.warning('Review collection failed: %s', exc)
        result.update({'status': 'error', 'message': str(exc)})
    return result

def collect_seller_information(seller_id, product_title='', marketplace_domain=''):
    if tavily_client is None:
        return {'seller_id': seller_id, 'status': 'unavailable', 'message': 'Seller external search unavailable'}
    query = f'{seller_id} {product_title} {marketplace_domain} seller reputation public listings reviews'
    search = search_web(query, max_results=5)
    return {'seller_id': seller_id, 'status': search.get('status', 'unavailable'), 'results': search.get('results', []), 'message': search.get('message')}

def find_related_listings(product_title, marketplace_domain, exclude_url=''):
    query = f'{product_title} {marketplace_domain} site:{marketplace_domain}'
    search = search_web(query, max_results=5)
    hits = search.get('results', [])
    listings = [{'url': h.get('url') or h.get('link'), 'title': h.get('title')} for h in hits if (h.get('url') or h.get('link')) and (h.get('url') or h.get('link')) != exclude_url]
    return {'product_title': product_title, 'marketplace_domain': marketplace_domain, 'listings': listings, 'status': search.get('status', 'unavailable')}


## 06. Product Extraction

In [6]:

# @title 06. Product Extraction
def extract_product_snapshot(product_page):
    data = product_page.get('data', {}) or {}
    content = data.get('markdown') or data.get('html') or ''
    title = ''
    if isinstance(content, str):
        m = re.search(r'<title[^>]*>(.*?)</title>', content, re.I | re.S)
        if m:
            title = re.sub(r'<.*?>', ' ', m.group(1))
            title = re.sub(r'\s+', ' ', title).strip()
    return {'title': title or 'Unknown product', 'url': product_page.get('url', ''), 'status': product_page.get('status', 'unavailable'), 'extracted_at': now_iso()}


## 07. Review Collection and Statistical Analyzer

In [7]:
# @title 07. Review Collection and Statistical Analyzer
def parse_review_texts(raw_text):
    reviews = []
    if not raw_text:
        return reviews
    for block in re.split(r'(?i)(?:\n\s*\n|\.|!|\?)', raw_text):
        b = (block or '').strip()
        if len(b) < 10:
            continue
        rating_match = re.search(r'(?:rating|stars?)\s*[:=]?\s*(\d+(?:\.\d+)?)', b, flags=re.I)
        rating = 5.0 if rating_match is None else safe_float(rating_match.group(1), 5.0)
        reviews.append({'review_id': f'R{len(reviews)+1:04d}', 'reviewer_id_or_name': 'public_reviewer', 'rating': max(1.0, min(5.0, rating)), 'review_text': b[:500], 'review_date': None, 'verified_purchase': None, 'helpful_votes': 0, 'product_id': None})
    return reviews

def review_statistical_analyzer(review_doc):
    reviews = review_doc.get('reviews', []) or []
    if not reviews:
        return {'score': 0.0, 'confidence': 0.05, 'signals': ['No public reviews available.'], 'metrics': {}, 'status': 'insufficient_data'}
    ratings = np.array([safe_float(r.get('rating'), 3.0) for r in reviews], dtype=float)
    star5_ratio = float(np.mean(ratings >= 4.5))
    seen = {}
    for review in reviews:
        text = (review.get('review_text') or '').strip().lower()
        if text:
            key = re.sub(r'\s+', ' ', text)
            seen[key] = seen.get(key, 0) + 1
    duplicate_texts = sum(1 for count in seen.values() if count > 1)
    duplicate_ratio = min(1.0, duplicate_texts / max(1, len(seen)))
    clf = IsolationForest(contamination=0.12, random_state=0)
    preds = clf.fit_predict(ratings.reshape(-1, 1))
    anomaly_rate = float(np.mean(preds == -1))
    raw = 0.35 * clamp01((star5_ratio - 0.6) / 0.4) + 0.30 * clamp01(min(1.0, len(reviews) / 50.0)) + 0.20 * clamp01(duplicate_ratio + anomaly_rate) + 0.15 * clamp01(len(reviews) / 200.0)
    return {'score': round(clamp01(raw), 4), 'confidence': 0.62, 'signals': ['Review concentration and public anomaly signals were observed.'], 'metrics': {'star5_ratio': round(star5_ratio, 4), 'review_count': len(reviews), 'duplicate_ratio': round(duplicate_ratio, 4), 'anomaly_rate': round(anomaly_rate, 4)}, 'status': 'ok'}


## 08. Linguistic Similarity Analyzer

In [8]:

# @title 08. Linguistic Similarity Analyzer
def linguistic_similarity_analyzer(review_doc):
    reviews = review_doc.get('reviews', []) or []
    if not reviews or sentence_model is None:
        return {'score': 0.0, 'similar_pairs': [], 'signals': ['Insufficient public review text or embedding model unavailable.'], 'status': 'insufficient_data'}
    texts = [(r.get('review_text') or '').strip() for r in reviews if (r.get('review_text') or '').strip()]
    if len(texts) < 2:
        return {'score': 0.0, 'similar_pairs': [], 'signals': ['Too few review texts.'], 'status': 'insufficient_data'}
    embeddings = sentence_model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings.astype('float32'))
    dists, idxs = index.search(embeddings.astype('float32'), 3)
    pairs = []
    for i in range(len(texts)):
        for j in range(1, 3):
            neighbor = int(idxs[i, j])
            sim = float(dists[i, j])
            if neighbor != i and sim >= 0.75:
                pairs.append({'review_a': i, 'review_b': neighbor, 'similarity': round(sim, 4), 'text_a': texts[i][:200], 'text_b': texts[neighbor][:200]})
    if not pairs:
        return {'score': 0.0, 'similar_pairs': [], 'signals': ['No highly similar review language detected.'], 'status': 'ok'}
    score = clamp01(np.mean([p['similarity'] for p in pairs]) * 1.2)
    return {'score': round(score, 4), 'similar_pairs': pairs[:20], 'signals': ['Multiple public reviews show strong semantic overlap.'], 'status': 'ok', 'explanation': 'Similar wording is an investigation signal, not proof of fake reviews.'}


## 09. Reviewer Network Analyzer

In [9]:

# @title 09. Reviewer Network Analyzer
def reviewer_network_analyzer(review_doc):
    reviews = review_doc.get('reviews', []) or []
    if not reviews:
        return {'score': 0.0, 'confidence': 0.0, 'signals': ['No public reviewer data.'], 'overlap': [], 'graph': {}, 'status': 'insufficient_cross_product_reviewer_data'}
    graph = nx.Graph()
    for review in reviews:
        reviewer = review.get('reviewer_id_or_name') or 'anonymous'
        product_id = review.get('product_id') or 'product-1'
        graph.add_node(reviewer, kind='reviewer')
        graph.add_node(product_id, kind='product')
        graph.add_edge(reviewer, product_id, relation='reviewed')
    overlap = [n for n in graph.nodes if graph.degree(n) > 1 and graph.nodes[n].get('kind') == 'reviewer']
    if not overlap:
        return {'score': 0.0, 'confidence': 0.05, 'signals': ['insufficient_cross_product_reviewer_data'], 'overlap': [], 'graph': {'nodes': list(graph.nodes()), 'edges': list(graph.edges())}, 'status': 'insufficient_cross_product_reviewer_data'}
    score = clamp01(len(overlap) / max(1, len(reviews)))
    return {'score': round(score, 4), 'confidence': 0.70, 'signals': ['Reviewer overlap observed across public product interactions.'], 'overlap': overlap[:20], 'graph': {'nodes': list(graph.nodes()), 'edges': list(graph.edges())}, 'status': 'ok'}


## 10. Claim Verification

In [10]:

# @title 10. Claim Verification
def claim_verification_agent(product_data):
    if tavily_client is None:
        return {'claim': 'No claim provided', 'verdict': 'uncertain', 'support_score': 0.0, 'reason': 'Tavily unavailable; claim verification could not be completed.', 'evidence': [], 'caveat': 'Claim support is evidence-based and can remain uncertain.'}
    claim_text = product_data.get('title', 'Product claim')
    search = search_web(f'{claim_text} official manufacturer specifications evidence', max_results=3)
    if not search.get('results'):
        return {'claim': claim_text, 'verdict': 'uncertain', 'support_score': 0.0, 'reason': 'No decisive external public evidence was found.', 'evidence': [], 'caveat': 'No claim support should not be treated as counterfeit proof.'}
    return {'claim': claim_text, 'verdict': 'partially_supported', 'support_score': 0.45, 'reason': 'Some public support exists, but not enough to fully validate the claim.', 'evidence': search.get('results', []), 'caveat': 'Support is limited and should be corroborated by a human investigator.'}


## 11. Visual Analysis

In [11]:

# @title 11. Visual Analysis
def embed_image(img):
    if clip_model is None or clip_preprocess is None:
        return None
    with torch.no_grad():
        tensor = clip_preprocess(img).unsqueeze(0).to(clip_device)
        emb = clip_model.encode_image(tensor)
        emb = emb / emb.norm(dim=-1, keepdim=True)
        return emb.cpu().numpy()[0].astype('float32')

def visual_similarity_agent(product_data, related_listings=None):
    images = product_data.get('images', []) or []
    if not images or clip_model is None:
        return {'score': 0.0, 'confidence': 0.0, 'matches': [], 'status': 'insufficient_comparison_images', 'message': 'No comparison images available.'}
    embeddings = []
    for img_url in images[:5]:
        try:
            resp = requests.get(img_url, timeout=20)
            if resp.ok:
                img = Image.open(BytesIO(resp.content)).convert('RGB')
                emb = embed_image(img)
                if emb is not None:
                    embeddings.append((img_url, emb))
        except Exception as exc:
            logger.warning('Image embedding failed: %s', exc)
    if not embeddings:
        return {'score': 0.0, 'confidence': 0.0, 'matches': [], 'status': 'insufficient_comparison_images', 'message': 'Could not create image embeddings.'}
    feats = np.stack([e for _, e in embeddings]).astype('float32')
    index = faiss.IndexFlatIP(feats.shape[1])
    index.add(feats)
    sims = index.search(feats, 2)[0]
    max_sim = float(np.max(sims[:, 1])) if sims.shape[1] > 1 else 0.0
    score = clamp01(max_sim * 1.2)
    return {'score': round(score, 4), 'confidence': 0.71, 'matches': [{'image_url': url, 'similarity': round(float(max_sim), 4)} for url, _ in embeddings[:3]], 'status': 'ok', 'message': 'Highly similar or reused product imagery detected.'}


## 12. Seller Risk Analyzer

In [12]:

# @title 12. Seller Risk Analyzer
def seller_risk_agent(seller_context, product_risk, review_risk):
    public_pages = seller_context.get('results', []) or []
    if not public_pages:
        return {'score': 0.15, 'confidence': 0.15, 'signals': ['No public seller data available; confidence reduced.'], 'observed_public_listing_pages': 0, 'status': 'insufficient_data'}
    score = clamp01(0.45 * product_risk.get('score', 0.0) + 0.55 * review_risk.get('score', 0.0) + 0.1)
    return {'score': round(score, 4), 'confidence': 0.62, 'signals': ['Public seller activity shows recurring pattern risk across listings.'], 'observed_public_listing_pages': len(public_pages), 'status': 'ok'}


## 13. Evidence Fusion and Compliance Mapping

In [13]:
# @title 13. Evidence Fusion and LLM-Grounded Compliance Mapping
def append_evidence(kind, finding, source, confidence, details=None):
    item = {'evidence_id': evidence_id(), 'type': kind, 'finding': finding, 'source': source, 'confidence': clamp01(confidence), 'details': details or {}, 'timestamp': now_iso()}
    state.evidence.append(item)
    return item['evidence_id']

def compliance_policy_kb():
    """Fixed policy taxonomy from the project brief; mapping to it is evidence-driven."""
    return [
        {'policy_id': 'P001', 'category': 'Review Manipulation', 'description': 'Potential suspicious review patterns, such as coordinated timing, repeated language, or reviewer overlap.'},
        {'policy_id': 'P002', 'category': 'Product Authenticity / Counterfeit Risk', 'description': 'Potential authenticity concerns requiring credible brand, manufacturer, or product-level corroboration.'},
        {'policy_id': 'P003', 'category': 'Product Misrepresentation', 'description': 'Potential material, origin, warranty, or other material product-claim inconsistency.'},
        {'policy_id': 'P004', 'category': 'Listing / Content Integrity', 'description': 'Potentially misleading or inconsistent listing text, images, or product information.'},
        {'policy_id': 'P005', 'category': 'Seller Integrity Risk', 'description': 'Potential recurring seller-level trust signals across observable public listings or activity.'},
    ]

def compliance_mapping_agent(evidence):
    """Ask Groq to map supplied evidence to the fixed policy KB and validate every returned ID."""
    policies = compliance_policy_kb()
    valid_evidence = {item.get('evidence_id'): item for item in evidence if item.get('evidence_id')}
    valid_policies = {policy['policy_id']: policy for policy in policies}
    if not valid_evidence:
        return {'status': 'ok', 'findings': [], 'unmapped_evidence_ids': [], 'error': None}

    tool = {
        'type': 'function',
        'function': {
            'name': 'submit_compliance_mappings',
            'description': 'Submit only conservative policy mappings supported by the evidence records provided in the prompt.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'mappings': {
                        'type': 'array',
                        'items': {
                            'type': 'object',
                            'properties': {
                                'evidence_id': {'type': 'string'},
                                'policy_id': {'type': 'string'},
                                'rationale': {'type': 'string'},
                                'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
                            },
                            'required': ['evidence_id', 'policy_id', 'rationale', 'confidence'],
                            'additionalProperties': False,
                        },
                    },
                    'unmapped_evidence_ids': {'type': 'array', 'items': {'type': 'string'}},
                },
                'required': ['mappings', 'unmapped_evidence_ids'],
                'additionalProperties': False,
            },
        },
    }
    # Send the observed evidence and policy descriptions only; the model must not create evidence or policies.
    prompt_payload = {
        'evidence': [
            {
                'evidence_id': item.get('evidence_id'),
                'type': item.get('type'),
                'finding': item.get('finding'),
                'confidence': item.get('confidence'),
                'details': item.get('details', {}),
                'source': item.get('source'),
            }
            for item in evidence
        ],
        'policy_categories': policies,
    }
    messages = [
        {
            'role': 'system',
            'content': (
                'You are a conservative Trust & Safety policy mapper. Map only supplied evidence to the supplied policy IDs. '
                'Return a mapping only when the evidence supports a potential concern; do not infer guilt, counterfeiting, '
                'or a confirmed policy violation. Do not create IDs, policies, facts, or legal rules. One evidence item may '
                'map to multiple categories only when separately justified. If evidence is insufficient, leave it unmapped.'
            ),
        },
        {'role': 'user', 'content': json.dumps(prompt_payload, ensure_ascii=False, default=str)},
    ]
    try:
        response = groq_chat(messages, tools=[tool], temperature=0.0)
        tool_calls = response.choices[0].message.tool_calls or []
        mapping_call = next((call for call in tool_calls if call.function.name == 'submit_compliance_mappings'), None)
        if mapping_call is None:
            raise ValueError('Groq did not return the required compliance-mapping tool call.')
        parsed = json.loads(mapping_call.function.arguments)
        findings = []
        seen = set()
        for mapping in parsed.get('mappings', []):
            evidence_id_value = mapping.get('evidence_id')
            policy_id_value = mapping.get('policy_id')
            if evidence_id_value not in valid_evidence or policy_id_value not in valid_policies:
                logger.warning('Discarded compliance mapping with unknown evidence or policy ID: %s / %s', evidence_id_value, policy_id_value)
                continue
            if (evidence_id_value, policy_id_value) in seen:
                continue
            rationale = str(mapping.get('rationale', '')).strip()
            if not rationale:
                continue
            policy = valid_policies[policy_id_value]
            findings.append({
                'evidence_id': evidence_id_value,
                'policy_id': policy_id_value,
                'category': policy['category'],
                'rationale': rationale,
                'confidence': clamp01(safe_float(mapping.get('confidence'), 0.0)),
            })
            seen.add((evidence_id_value, policy_id_value))
        mapped_ids = {finding['evidence_id'] for finding in findings}
        explicitly_unmapped = {item for item in parsed.get('unmapped_evidence_ids', []) if item in valid_evidence}
        unmapped_ids = sorted((set(valid_evidence) - mapped_ids) | explicitly_unmapped)
        return {'status': 'ok', 'findings': findings, 'unmapped_evidence_ids': unmapped_ids, 'error': None}
    except Exception as exc:
        logger.error('LLM compliance mapping failed: %s', exc)
        return {
            'status': 'unavailable',
            'findings': [],
            'unmapped_evidence_ids': sorted(valid_evidence),
            'error': str(exc),
        }


## 14. Deterministic Trust Risk Engine

In [14]:

# @title 14. Deterministic Trust Risk Engine
def risk_level_from_score(score):
    value = clamp01(score) * 100
    for label, low, high in RISK_LEVELS:
        if low <= value <= high:
            return label
    return 'LOW'

def calculate_trust_risk(score_map):
    weighted = {key: WEIGHTS[key] * clamp01(score_map.get(key, 0.0)) for key in WEIGHTS}
    raw = sum(weighted.values())
    percent = round(raw * 100.0, 2)
    return {
        'raw_score': round(raw, 4),
        'weight': WEIGHTS,
        'weighted_contribution': {k: round(v, 4) for k, v in weighted.items()},
        'final_risk': percent,
        'risk_level': risk_level_from_score(raw),
        'formula': 'R = Σ(wᵢ × sᵢ)',
        'thresholds': {'LOW': '0–30', 'MEDIUM': '31–60', 'HIGH': '61–80', 'CRITICAL': '81–100'}
    }


## 15. Master Investigation Function

In [15]:
# @title 15. Master Investigation Function
def _show_stage(stage_name, payload):
    """Print a readable, JSON-safe checkpoint for one investigation stage."""
    print(f"\n{'=' * 18} {stage_name} {'=' * 18}")
    try:
        print(json.dumps(payload, ensure_ascii=False, indent=2, default=str))
    except Exception as exc:
        logger.warning('Could not display stage %s as JSON: %s', stage_name, exc)
        print(str(payload))

def investigate(product_url, seller_id, snapshot_mode=False, verbose=True, progress_callback=None):
    """Run a public-evidence investigation and optionally print every pipeline stage."""
    state.evidence = []
    pipeline_trace = []

    def record(stage_name, payload):
        stage = {'stage': stage_name, 'timestamp': now_iso(), 'output': payload}
        pipeline_trace.append(stage)
        if verbose:
            _show_stage(stage_name, payload)
        if progress_callback is not None:
            progress_callback(stage_name, payload)
        return payload

    if verbose:
        print('PS-13 investigation starting. Collection and analyses run sequentially; unavailable data is reported, not inferred.')
    record('Input', {'product_url': product_url, 'seller_id': seller_id, 'started_at': now_iso()})

    plan = orchestrator_decide(product_url, seller_id)
    record('Orchestrator plan', {'selected_tools': plan, 'note': 'The investigation still runs required collection/analysis stages even if planning is unavailable.'})

    product_page = collect_product_page(product_url)
    product_snapshot = extract_product_snapshot(product_page)
    record('Product page collection', {
        'status': product_page.get('status'),
        'method': product_page.get('method'),
        'source_url': product_page.get('url', product_url),
        'fallback_note': product_page.get('fallback_note'),
        'message': product_page.get('message'),
        'product_snapshot': product_snapshot,
        'content_available': bool((product_page.get('data') or {}).get('markdown') or (product_page.get('data') or {}).get('html')),
    })

    review_doc = collect_reviews(product_url)
    if review_doc.get('raw_content'):
        review_doc['reviews'] = parse_review_texts(review_doc['raw_content'])
    else:
        review_doc['reviews'] = []
    record('Review collection', {
        'status': review_doc.get('status'),
        'source_url': review_doc.get('source_url', product_url),
        'extraction_method': review_doc.get('extraction_method'),
        'coverage_notes': review_doc.get('coverage_notes', []),
        'review_count_parsed': len(review_doc['reviews']),
        'message': review_doc.get('message'),
        'sample_reviews': review_doc['reviews'][:3],
    })

    statistical = review_statistical_analyzer(review_doc)
    record('Agent 2 ? Statistical review analysis', statistical)

    linguistic = linguistic_similarity_analyzer(review_doc)
    record('Agent 3 ? Linguistic similarity', linguistic)

    network = reviewer_network_analyzer(review_doc)
    record('Agent 4 ? Reviewer network', network)

    related = find_related_listings(product_snapshot.get('title', 'Unknown product'), urlparse(product_url).netloc, exclude_url=product_url)
    record('Related public listings', related)

    claim = claim_verification_agent({'title': product_snapshot.get('title', 'Product claim')})
    record('Agent 5 ? Claim verification', claim)

    visual = visual_similarity_agent(product_snapshot, related.get('listings', []))
    record('Agent 6 ? Visual similarity', visual)

    seller_context = collect_seller_information(seller_id, product_snapshot.get('title', ''), urlparse(product_url).netloc)
    seller = seller_risk_agent(seller_context, claim, statistical)
    record('Agent 7 ? Seller risk', {'public_search': seller_context, 'analysis': seller})

    evidence_before = len(state.evidence)
    for payload, kind, source, confidence, finding in [
        (statistical, 'review_statistical', product_url, statistical.get('confidence', 0.0), 'Review statistics show concentration, volume, or anomaly signals.'),
        (linguistic, 'linguistic_similarity', product_url, linguistic.get('score', 0.0), 'Review language similarity was analyzed; see details for observed pairs and limitations.'),
        (network, 'reviewer_network', product_url, network.get('confidence', 0.0), 'Public reviewer-network analysis was performed; see status and graph for available evidence.'),
        (claim, 'claim_verification', product_url, 1.0 - safe_float(claim.get('support_score', 0.0), 0.0), 'Product claim support was assessed using available external evidence.'),
        (visual, 'visual_similarity', product_url, visual.get('confidence', 0.0), 'Visual comparison status and any available matches are recorded in details.'),
        (seller, 'seller_risk', product_url, seller.get('confidence', 0.0), 'Seller-level public evidence was assessed; see details and coverage.'),
    ]:
        if payload.get('status') not in ('insufficient_data', 'insufficient_cross_product_reviewer_data', 'insufficient_comparison_images'):
            append_evidence(kind, finding, source, confidence, payload)
    record('Evidence fusion', {'new_evidence_count': len(state.evidence) - evidence_before, 'evidence_items': state.evidence})

    compliance_result = compliance_mapping_agent(state.evidence)
    compliance_findings = compliance_result['findings']
    record('Agent 8 ? LLM compliance mapping', {
        **compliance_result,
        'policy_categories': compliance_policy_kb(),
        'note': 'Mappings are potential policy concerns, not confirmed violations. An unavailable mapper or empty list is not proof that no concern exists.',
    })

    score_map = {
        'statistical': clamp01(statistical.get('score', 0.0)),
        'linguistic': clamp01(linguistic.get('score', 0.0)),
        'reviewer_network': clamp01(network.get('score', 0.0)),
        'claim_risk': clamp01(1.0 - safe_float(claim.get('support_score', 0.0), 0.0)),
        'visual': clamp01(visual.get('score', 0.0)),
        'seller_risk': clamp01(seller.get('score', 0.0)),
    }
    risk_summary = calculate_trust_risk(score_map)
    record('Agent 9 ? Deterministic risk calculation', {'input_scores': score_map, **risk_summary})

    report = {
        'product_url': product_url,
        'seller_id': seller_id,
        'executive_summary': 'Public evidence indicates a trust-risk investigation. No automatic enforcement is recommended.',
        'risk_score': int(round(risk_summary['final_risk'])),
        'risk_level': risk_summary['risk_level'],
        'key_findings': [item['finding'] for item in state.evidence[:5]],
        'evidence_summary': [item['finding'] for item in state.evidence[:5]],
        'compliance_findings': [f"{c['category']} ({c['policy_id']}; evidence {c['evidence_id']})" for c in compliance_findings],
        'compliance_mapping_status': compliance_result['status'],
        'recommended_action': 'Escalate to a human trust and safety investigator for manual review and corroboration before any enforcement action.',
        'limitations': ['Public evidence is limited to what is visible on the web.', 'The system does not confirm fraud or counterfeit without human review.', 'Collection failures or missing public data reduce the strength of conclusions.'],
    }
    record('Agent 10 ? Final report', report)
    if verbose:
        print('\nInvestigation stages complete. See result["pipeline_trace"] for the captured stage-by-stage outputs.')
    return {
        'product_url': product_url,
        'seller_id': seller_id,
        'product_snapshot': product_snapshot,
        'review_analysis': statistical,
        'linguistic_analysis': linguistic,
        'network_analysis': network,
        'claim_verification': claim,
        'visual_analysis': visual,
        'seller_risk': seller,
        'evidence': state.evidence,
        'compliance_findings': compliance_findings,
        'compliance_mapping': compliance_result,
        'risk_summary': risk_summary,
        'pipeline_trace': pipeline_trace,
        'report': report,
        'status': 'ok' if state.evidence else 'limited',
    }

# Example:
# result = investigate(product_url='https://example.com/product/12345', seller_id='SELLER-501', verbose=True)
# print(json.dumps(result['report'], indent=2))
# print(json.dumps(result['pipeline_trace'], indent=2, default=str))


## 16. Stage Output, Final Report, and Save Options

`investigate(..., verbose=True)` prints each stage as it runs. The returned `pipeline_trace` stores the same stage outputs. Set `verbose=False` to suppress progress output.


In [16]:
# @title 16. Final Report Display and Save Options
def save_json_report(payload, path='ps13_report.json'):
    with open(path, 'w', encoding='utf-8') as fp:
        json.dump(payload, fp, ensure_ascii=False, indent=2)
    return path

def save_markdown_report(payload, path='ps13_report.md'):
    report = payload.get('report', {})
    lines = [
        '# PS-13 Marketplace Trust Investigation',
        '',
        f"Product URL: {report.get('product_url', 'N/A')}",
        f"Seller ID: {report.get('seller_id', 'N/A')}",
        f"Risk Score: {report.get('risk_score', 'N/A')} / 100",
        f"Risk Level: {report.get('risk_level', 'N/A')}",
        '',
        '## Executive Summary',
        report.get('executive_summary', ''),
        '',
        '## Key Findings',
    ]
    for item in report.get('key_findings', []) or []:
        lines.append(f'- {item}')
    lines.extend(['', '## Compliance Findings'])
    for item in report.get('compliance_findings', []) or []:
        lines.append(f'- {item}')
    with open(path, 'w', encoding='utf-8') as fp:
        fp.write('\n'.join(lines))
    return path

print('PS-13 AGENT READY')


PS-13 AGENT READY


## 17. Streamlit Web Interface

The final launch cell creates a browser-based Streamlit interface with live stage progress, evidence and score tabs, report downloads, Colab access protection, and the same `investigate()` pipeline. Execute the notebook top-to-bottom, then run the launch cell. The app runs only while the Colab runtime remains active.


In [19]:
# @title 17. Launch PS-13 Streamlit Outside Colab Browser - GROQ

import os
import sys
import time
import json
import hmac
import secrets
import shutil
import subprocess
import urllib.request
from pathlib import Path
from IPython import get_ipython


# ============================================================
# 1. STREAMLIT APP
# ============================================================

PS13_STREAMLIT_APP_SOURCE = r'''
import json
import os
import hmac

import pandas as pd
import streamlit as st
import ps13_core


st.set_page_config(
    page_title="PS-13 Trust Investigation",
    page_icon="🔎",
    layout="wide",
)


# ============================================================
# CSS
# ============================================================

st.markdown("""
<style>

.block-container {
    max-width: 1180px;
    padding-top: 1.8rem;
    padding-bottom: 3rem;
}

[data-testid="stMetric"] {
    background: #f5f7fb;
    border: 1px solid #e3e8f0;
    padding: 14px;
    border-radius: 12px;
}

</style>
""", unsafe_allow_html=True)


# ============================================================
# HEADER
# ============================================================

st.title("🔎 PS-13 Marketplace Trust Investigation")

st.caption(
    "Public-evidence decision support for human investigators — "
    "not a counterfeit verdict or automated enforcement system."
)


# ============================================================
# ACCESS CODE
# ============================================================

access_code = os.environ.get(
    "PS13_UI_ACCESS_CODE"
)

if access_code:

    supplied = st.sidebar.text_input(
        "Colab session access code",
        type="password",
    )

    if not hmac.compare_digest(
        supplied,
        access_code
    ):

        st.info(
            "Enter the access code printed by the Colab cell."
        )

        st.stop()


# ============================================================
# SIDEBAR
# ============================================================

with st.sidebar:

    st.subheader("Investigation setup")

    st.caption(
        "Only public product information is collected. "
        "Never enter passwords, private order data, "
        "serial numbers, or API keys here."
    )

    st.markdown("**Service status**")

    # --------------------------------------------------------
    # GROQ
    # --------------------------------------------------------

    st.write(
        "Groq:",
        "Configured"
        if os.getenv("GROQ_API_KEY")
        else "Missing",
    )

    # --------------------------------------------------------
    # TAVILY
    # --------------------------------------------------------

    st.write(
        "Tavily:",
        "Configured"
        if os.getenv("TAVILY_API_KEY")
        else "Unavailable",
    )

    # --------------------------------------------------------
    # FIRECRAWL
    # --------------------------------------------------------

    st.write(
        "Firecrawl:",
        "Configured"
        if os.getenv("FIRECRAWL_API_KEY")
        else "Playwright fallback",
    )


# ============================================================
# INPUT FORM
# ============================================================

with st.form(
    "investigation_form",
    clear_on_submit=False
):

    product_url = st.text_input(
        "Public product listing URL",
        placeholder="https://www.amazon.com/dp/XXXXXXXXXX",
    )

    seller_id = st.text_input(
        "Seller ID / Seller Name",
        placeholder="Enter seller name or seller identifier",
    )

    submitted = st.form_submit_button(
        "Run investigation",
        type="primary",
        use_container_width=True,
    )


# ============================================================
# INVESTIGATION
# ============================================================

if submitted:

    if not product_url.startswith(
        ("https://", "http://")
    ):

        st.error(
            "Enter a complete public URL beginning with "
            "http:// or https://."
        )

    elif not seller_id.strip():

        st.error(
            "Enter the seller name or seller ID shown on the listing."
        )

    else:

        stage_box = st.container()

        progress = st.progress(
            0,
            text="Preparing investigation..."
        )

        stage_total = 14

        seen_stages = [0]


        with st.status(
            "Investigation running",
            expanded=True
        ) as status:


            def on_stage(
                name,
                payload
            ):

                seen_stages[0] += 1

                progress.progress(
                    min(
                        seen_stages[0] / stage_total,
                        0.99
                    ),
                    text=f"Completed: {name}",
                )

                with stage_box.expander(
                    name,
                    expanded=(
                        seen_stages[0] <= 2
                    ),
                ):

                    st.json(
                        payload,
                        expanded=False
                    )


            try:

                result = ps13_core.investigate(

                    product_url=product_url.strip(),

                    seller_id=seller_id.strip(),

                    verbose=False,

                    progress_callback=on_stage,
                )


                st.session_state[
                    "ps13_result"
                ] = result


                status.update(
                    label="Investigation completed",
                    state="complete",
                    expanded=False,
                )


                progress.progress(
                    1.0,
                    text="All available stages completed",
                )


            except Exception as exc:

                status.update(
                    label="Investigation failed",
                    state="error",
                    expanded=True,
                )

                st.exception(exc)


# ============================================================
# RESULTS
# ============================================================

result = st.session_state.get(
    "ps13_result"
)


if result:

    report = result.get(
        "report",
        {}
    )


    st.divider()

    st.subheader(
        "Investigation result"
    )


    if result.get(
        "status"
    ) == "limited":

        st.warning(
            "Evidence collection was limited. "
            "Do not interpret missing evidence as a "
            "low-risk or genuine-product finding."
        )


    # ========================================================
    # METRICS
    # ========================================================

    c1, c2, c3 = st.columns(3)


    c1.metric(
        "Trust risk",
        f"{report.get('risk_score', 'N/A')} / 100"
    )


    c2.metric(
        "Risk band",
        report.get(
            "risk_level",
            "Unavailable"
        )
    )


    c3.metric(
        "Evidence items",
        len(
            result.get(
                "evidence",
                []
            )
        )
    )


    st.info(
        report.get(
            "executive_summary",
            "No summary available."
        )
    )


    # ========================================================
    # TABS
    # ========================================================

    (
        tab_summary,
        tab_stages,
        tab_evidence,
        tab_scores,
        tab_export
    ) = st.tabs(
        [
            "Summary",
            "Pipeline stages",
            "Evidence",
            "Score breakdown",
            "Export",
        ]
    )


    # ========================================================
    # SUMMARY
    # ========================================================

    with tab_summary:

        st.markdown(
            "**Recommended human review**"
        )

        st.write(
            report.get(
                "recommended_action",
                "No recommendation available.",
            )
        )


        st.markdown(
            "**Compliance categories**"
        )

        categories = report.get(
            "compliance_findings",
            []
        )


        if categories:

            for item in categories:

                st.write(
                    "•",
                    item
                )

        else:

            st.write(
                "No categories were returned. "
                "Check compliance mapping status "
                "and evidence coverage; empty does "
                "not mean no concern."
            )


        st.markdown(
            "**Limitations**"
        )


        for limitation in report.get(
            "limitations",
            []
        ):

            st.write(
                "•",
                limitation
            )


    # ========================================================
    # PIPELINE
    # ========================================================

    with tab_stages:

        for stage in result.get(
            "pipeline_trace",
            []
        ):

            with st.expander(
                stage.get(
                    "stage",
                    "Stage"
                ),
                expanded=False
            ):

                st.caption(
                    stage.get(
                        "timestamp",
                        ""
                    )
                )

                st.json(
                    stage.get(
                        "output",
                        {}
                    ),
                    expanded=False
                )


    # ========================================================
    # EVIDENCE
    # ========================================================

    with tab_evidence:

        evidence = result.get(
            "evidence",
            []
        )


        if evidence:

            st.dataframe(
                pd.DataFrame(evidence),
                use_container_width=True,
                hide_index=True,
            )

        else:

            st.warning(
                "No evidence items were recorded. "
                "Collection may have been blocked "
                "or unavailable."
            )


        mapping = result.get(
            "compliance_mapping",
            {}
        )


        st.caption(
            "Compliance mapper status: "
            f"{mapping.get('status', 'not recorded')}"
        )


        if mapping.get(
            "error"
        ):

            st.error(
                mapping["error"]
            )


    # ========================================================
    # SCORE BREAKDOWN
    # ========================================================

    with tab_scores:

        summary = result.get(
            "risk_summary",
            {}
        )


        weights = summary.get(
            "weight",
            {}
        )


        contributions = summary.get(
            "weighted_contribution",
            {}
        )


        rows = [

            {
                "Signal":
                    key.replace(
                        "_",
                        " "
                    ).title(),

                "Weight":
                    weight,

                "Weighted contribution (0–1)":
                    contributions.get(
                        key,
                        0.0
                    ),
            }

            for key, weight
            in weights.items()

        ]


        if rows:

            st.dataframe(
                pd.DataFrame(rows),
                use_container_width=True,
                hide_index=True,
            )


        st.write(
            "Formula:",
            summary.get(
                "formula",
                "R = Σ(wᵢ × sᵢ)"
            )
        )


        st.json(
            summary
        )


    # ========================================================
    # EXPORT
    # ========================================================

    with tab_export:

        json_bytes = json.dumps(
            result,
            ensure_ascii=False,
            indent=2,
            default=str,
        ).encode("utf-8")


        st.download_button(
            "Download full result (JSON)",
            data=json_bytes,
            file_name="ps13_investigation.json",
            mime="application/json",
        )


        md_lines = [

            "# PS-13 Marketplace Trust Investigation",

            "",

            f"- Product URL: "
            f"{report.get('product_url', '')}",

            f"- Seller ID / Name: "
            f"{report.get('seller_id', '')}",

            (
                f"- Risk: "
                f"{report.get('risk_score', '')}/100 "
                f"({report.get('risk_level', '')})"
            ),

            "",

            "## Executive summary",

            report.get(
                "executive_summary",
                ""
            ),

            "",

            "## Key findings",

            *[
                f"- {item}"

                for item in report.get(
                    "key_findings",
                    []
                )
            ],

            "",

            "## Compliance findings",

            *[
                f"- {item}"

                for item in report.get(
                    "compliance_findings",
                    []
                )
            ],

            "",

            "## Limitations",

            *[
                f"- {item}"

                for item in report.get(
                    "limitations",
                    []
                )
            ],

        ]


        st.download_button(
            "Download report (Markdown)",
            data="\n".join(md_lines),
            file_name="ps13_investigation.md",
            mime="text/markdown",
        )


else:

    st.markdown(
        "### What this app does"
    )


    st.write(
        "Collects available public listing data, "
        "runs review, product, and seller analyses, "
        "shows intermediate outputs, and produces "
        "an explainable risk summary."
    )


    st.warning(
        "Results are evidence signals for a human "
        "investigator. They do not prove that a "
        "product is fake or that a seller violated policy."
    )

'''


# ============================================================
# 2. INSTALL STREAMLIT
# ============================================================

print("Installing/checking Streamlit...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "streamlit",
    ],
    check=True,
)

print("Streamlit ready.")


# ============================================================
# 3. INSTALL CLOUDFLARED
# ============================================================

print("Checking Cloudflare Tunnel...")

cloudflared = shutil.which(
    "cloudflared"
)


if cloudflared is None:

    print(
        "Cloudflare Tunnel not found. Installing..."
    )

    subprocess.run(
        [
            "bash",
            "-lc",
            """
            wget -q \
              https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
              -O /usr/local/bin/cloudflared

            chmod +x /usr/local/bin/cloudflared
            """,
        ],
        check=True,
    )

    cloudflared = shutil.which(
        "cloudflared"
    )


if cloudflared is None:

    raise RuntimeError(
        "Could not install cloudflared."
    )


print(
    "Cloudflare Tunnel ready:",
    cloudflared
)


# ============================================================
# 4. GET COLAB HISTORY
# ============================================================

ip = get_ipython()


if ip is None:

    raise RuntimeError(
        "This cell must be executed inside Google Colab."
    )


history = ip.user_ns.get(
    "In",
    []
)


# ============================================================
# 5. FIND PS-13 CODE
# ============================================================

selected = []


for cell in history:

    if not isinstance(
        cell,
        str
    ):
        continue


    if not cell.strip():
        continue


    # --------------------------------------------------------
    # NEVER COPY THIS LAUNCH CELL INTO ps13_core.py
    # --------------------------------------------------------

    launch_markers = [

        "PS13_STREAMLIT_APP_SOURCE",

        "Cloudflare Tunnel",

        "PS13_STREAMLIT_PROCESS",

        "PS13_CLOUDFLARE_PROCESS",

        "cloudflared",

        "ps13_streamlit.log",

        "ps13_cloudflare.log",

    ]


    marker_count = sum(
        marker in cell
        for marker in launch_markers
    )


    if marker_count >= 2:

        continue


    # --------------------------------------------------------
    # KEEP ACTUAL PS-13 IMPLEMENTATION CELLS
    # --------------------------------------------------------

    implementation_markers = [

        "def ",

        "class ",

        "from openai import",

        "import openai",

        "import pandas",

        "import numpy",

        "import requests",

        "import json",

        "import os",

        "import re",

        "import math",

        "import statistics",

        "import networkx",

        "import sklearn",

        "import torch",

        "import PIL",

        "from PIL",

        "from sentence_transformers",

        "from bs4",

        "from urllib",

        "import urllib",

        "import httpx",

    ]


    if any(
        marker in cell
        for marker in implementation_markers
    ):

        selected.append(
            cell
        )


# ============================================================
# 6. CHECK
# ============================================================

if not selected:

    raise RuntimeError(
        "No PS-13 implementation cells were found. "
        "Run your PS-13 analysis cells first."
    )


print(
    f"Collected {len(selected)} executed code cells."
)


# ============================================================
# 7. CREATE CORE FILE
# ============================================================

workspace = Path(
    "/content"
)


core_path = (
    workspace /
    "ps13_core.py"
)


app_path = (
    workspace /
    "ps13_app.py"
)


core_source = "\n\n".join(
    selected
)


# ------------------------------------------------------------
# SAFETY CHECK
# ------------------------------------------------------------

for forbidden in [

    "serve_kernel_port_as_iframe",

    "PS13_STREAMLIT_APP_SOURCE",

    "PS13_STREAMLIT_PROCESS",

    "PS13_CLOUDFLARE_PROCESS",

    "cloudflared tunnel",

]:

    if forbidden in core_source:

        raise RuntimeError(
            "The Streamlit launcher was accidentally "
            "included in ps13_core.py."
        )


core_path.write_text(
    core_source,
    encoding="utf-8"
)


app_path.write_text(
    PS13_STREAMLIT_APP_SOURCE,
    encoding="utf-8"
)


print(
    "Created:",
    core_path
)

print(
    "Created:",
    app_path
)


# ============================================================
# 8. ACCESS CODE
# ============================================================

access_code = secrets.token_urlsafe(
    12
)


env = os.environ.copy()


env[
    "PS13_UI_ACCESS_CODE"
] = access_code


# ============================================================
# 9. STOP PREVIOUS STREAMLIT
# ============================================================

previous_streamlit = globals().get(
    "PS13_STREAMLIT_PROCESS"
)


if (
    previous_streamlit is not None
    and previous_streamlit.poll() is None
):

    print(
        "Stopping previous Streamlit process..."
    )


    previous_streamlit.terminate()


    try:

        previous_streamlit.wait(
            timeout=10
        )

    except subprocess.TimeoutExpired:

        previous_streamlit.kill()


# ============================================================
# 10. START STREAMLIT
# ============================================================

log_path = (
    workspace /
    "ps13_streamlit.log"
)


log_file = open(
    log_path,
    "w",
    encoding="utf-8"
)


print(
    "Starting Streamlit on port 8501..."
)


PS13_STREAMLIT_PROCESS = subprocess.Popen(

    [
        sys.executable,

        "-m",

        "streamlit",

        "run",

        str(app_path),

        "--server.port",

        "8501",

        "--server.address",

        "0.0.0.0",

        "--server.headless",

        "true",

        "--server.enableCORS",

        "false",

        "--server.enableXsrfProtection",

        "false",

        "--server.enableWebsocketCompression",

        "false",

    ],

    cwd=str(
        workspace
    ),

    env=env,

    stdout=log_file,

    stderr=subprocess.STDOUT,

)


# ============================================================
# 11. WAIT FOR STREAMLIT
# ============================================================

print(
    "Waiting for Streamlit..."
)


ready = False


for _ in range(45):

    if (
        PS13_STREAMLIT_PROCESS.poll()
        is not None
    ):

        log_file.flush()

        log_text = log_path.read_text(
            encoding="utf-8"
        )

        raise RuntimeError(
            "Streamlit exited during startup.\n\n"
            + log_text
        )


    try:

        with urllib.request.urlopen(
            "http://127.0.0.1:8501/_stcore/health",
            timeout=2,
        ) as response:

            if response.status == 200:

                ready = True

                break


    except Exception:

        time.sleep(1)


if not ready:

    log_file.flush()

    log_text = log_path.read_text(
        encoding="utf-8"
    )

    raise RuntimeError(
        "Streamlit did not become ready.\n\n"
        + log_text
    )


print()
print(
    "=============================================="
)

print(
    "STREAMLIT SERVER IS RUNNING"
)

print(
    "=============================================="
)

print(
    "Local port: 8501"
)

print(
    "Health check: OK"
)

print(
    "PID:",
    PS13_STREAMLIT_PROCESS.pid
)

print(
    "=============================================="
)


# ============================================================
# 12. STOP PREVIOUS CLOUDFLARE TUNNEL
# ============================================================

previous_tunnel = globals().get(
    "PS13_CLOUDFLARE_PROCESS"
)


if (
    previous_tunnel is not None
    and previous_tunnel.poll() is None
):

    print(
        "Stopping previous Cloudflare tunnel..."
    )


    previous_tunnel.terminate()


    try:

        previous_tunnel.wait(
            timeout=10
        )

    except subprocess.TimeoutExpired:

        previous_tunnel.kill()


# ============================================================
# 13. START CLOUDFLARE QUICK TUNNEL
# ============================================================

tunnel_log_path = (
    workspace /
    "ps13_cloudflare.log"
)


tunnel_log = open(
    tunnel_log_path,
    "w",
    encoding="utf-8"
)


print()
print(
    "Starting public Cloudflare Tunnel..."
)


PS13_CLOUDFLARE_PROCESS = subprocess.Popen(

    [

        cloudflared,

        "tunnel",

        "--url",

        "http://127.0.0.1:8501",

        "--no-autoupdate",

    ],

    stdout=tunnel_log,

    stderr=subprocess.STDOUT,

    env=env,

)


# ============================================================
# 14. WAIT FOR PUBLIC URL
# ============================================================

public_url = None


for _ in range(40):

    if (
        PS13_CLOUDFLARE_PROCESS.poll()
        is not None
    ):

        tunnel_log.flush()

        tunnel_text = tunnel_log_path.read_text(
            encoding="utf-8"
        )

        raise RuntimeError(
            "Cloudflare Tunnel stopped.\n\n"
            + tunnel_text
        )


    time.sleep(1)

    tunnel_log.flush()


    try:

        tunnel_text = tunnel_log_path.read_text(
            encoding="utf-8"
        )


        import re


        urls = re.findall(
            r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com",
            tunnel_text
        )


        if urls:

            public_url = urls[-1]

            break


    except Exception:

        pass


# ============================================================
# 15. PUBLIC URL
# ============================================================

if not public_url:

    tunnel_log.flush()

    tunnel_text = tunnel_log_path.read_text(
        encoding="utf-8"
    )

    raise RuntimeError(

        "Cloudflare did not provide a public URL "
        "within the expected time.\n\n"

        "Tunnel log:\n"

        + tunnel_text

    )


print()
print()

print(
    "============================================================"
)

print(
    "🚀 PS-13 STREAMLIT APP IS LIVE"
)

print(
    "============================================================"
)

print()

print(
    "OPEN THIS URL IN CHROME / FIREFOX:"
)

print()

print(
    public_url
)

print()

print(
    "============================================================"
)

print(
    "ACCESS CODE:"
)

print(
    access_code
)

print(
    "============================================================"
)

print()

print(
    "IMPORTANT:"
)

print(
    "Keep this Colab runtime running."
)

print(
    "The public URL works only while this Colab session "
    "and the Streamlit process are running."
)

print()

print(
    "Streamlit log:",
    log_path
)

print(
    "Cloudflare log:",
    tunnel_log_path
)

print(
    "============================================================"
)

Installing/checking Streamlit...
Streamlit ready.
Checking Cloudflare Tunnel...
Cloudflare Tunnel ready: /usr/local/bin/cloudflared
Collected 16 executed code cells.
Created: /content/ps13_core.py
Created: /content/ps13_app.py
Stopping previous Streamlit process...
Starting Streamlit on port 8501...
Waiting for Streamlit...

STREAMLIT SERVER IS RUNNING
Local port: 8501
Health check: OK
PID: 6076
Stopping previous Cloudflare tunnel...

Starting public Cloudflare Tunnel...


🚀 PS-13 STREAMLIT APP IS LIVE

OPEN THIS URL IN CHROME / FIREFOX:

https://wind-conversations-lou-autos.trycloudflare.com

ACCESS CODE:
qfC2owuJrOJF1im7

IMPORTANT:
Keep this Colab runtime running.
The public URL works only while this Colab session and the Streamlit process are running.

Streamlit log: /content/ps13_streamlit.log
Cloudflare log: /content/ps13_cloudflare.log


### Final note

This notebook is intentionally modular so each agent can be replaced independently later. It emphasizes correctness, traceability, explainability, graceful failure, and conservative wording rather than forcing a conclusion from weak evidence.

In [18]:
def orchestrator_decide(product_url, seller_id):
    prompt = f"""
    You are the marketplace trust orchestrator.
    Decide which public collection actions are needed for: {product_url}, seller_id={seller_id}.
    Allowed tools: ['collect_product_page', 'collect_reviews', 'search_web',
                    'collect_seller_information', 'find_related_listings'].
    Return either a JSON array of allowed tool-name strings or a JSON object
    with a 'tools' array. Do not invent evidence.
    """
    allowed_tools = {
        "collect_product_page",
        "collect_reviews",
        "search_web",
        "collect_seller_information",
        "find_related_listings",
    }

    result = groq_extract_json(prompt, "tool plan", fallback=[])

    if isinstance(result, dict):
        requested = result.get("tools") or result.get("tool_names") or []
    elif isinstance(result, list):
        requested = result
    else:
        logger.warning(
            "Groq returned an unsupported orchestration plan type: %s",
            type(result).__name__,
        )
        return []

    if not isinstance(requested, list):
        logger.warning("Groq orchestration plan did not contain a list of tools.")
        return []

    return [
        name for name in requested
        if isinstance(name, str) and name in allowed_tools
    ]

##inputs


In [20]:
#PRODUCT 1
result = investigate(
    product_url="https://www.flipkart.com/apple-iphone-16-black-128-gb/p/itmb07d67f995271?pid=MOBH4DQFG8NKFRDY",
    seller_id="f6d15b4a4f304426"
)
print(json.dumps(result["report"], indent=2))

PS-13 investigation starting. Collection and analyses run sequentially; unavailable data is reported, not inferred.

================== Input ==================
{
  "product_url": "https://www.flipkart.com/apple-iphone-16-black-128-gb/p/itmb07d67f995271?pid=MOBH4DQFG8NKFRDY",
  "seller_id": "f6d15b4a4f304426",
  "started_at": "2026-10-07T06:32:33.516763+00:00"
}

================== Orchestrator plan ==================
{
  "selected_tools": [
    "collect_product_page",
    "collect_seller_information",
    "collect_reviews"
  ],
  "note": "The investigation still runs required collection/analysis stages even if planning is unavailable."
}

================== Product page collection ==================
{
  "status": "ok",
  "method": "firecrawl",
  "source_url": "https://www.flipkart.com/apple-iphone-16-black-128-gb/p/itmb07d67f995271?pid=MOBH4DQFG8NKFRDY",
  "fallback_note": null,
  "message": null,
  "product_snapshot": {
    "title": "Unknown product",
    "url": "https://www.flipkart

In [ ]:
# PRODUCT 2

result = investigate(
    product_url="https://www.amazon.com/LEGO-at-ST-Walker-Puzzle-Adults/dp/B0DRW6CYGQ/ref=pd_rhf_se_s_pd_sbs_rvi_d_sccl_1_8/133-6169972-1441855?pd_rd_w=pDHm4&content-id=amzn1.sym.6640a844-ab24-4352-ac9b-78899e683a5e&pf_rd_p=6640a844-ab24-4352-ac9b-78899e683a5e&pf_rd_r=SZAWKWXH9GKEB7FK79DX&pd_rd_wg=zuRSF&pd_rd_r=55c3032c-b91e-47b5-85ec-fe5368334871&pd_rd_i=B0DRW6CYGQ&psc=1",
    seller_id="Amazon.com"
)

print(json.dumps(result["report"], indent=2))

PS-13 investigation starting. Collection and analyses run sequentially; unavailable data is reported, not inferred.

================== Input ==================
{
  "product_url": "https://www.amazon.com/LEGO-at-ST-Walker-Puzzle-Adults/dp/B0DRW6CYGQ/ref=pd_rhf_se_s_pd_sbs_rvi_d_sccl_1_8/133-6169972-1441855?pd_rd_w=pDHm4&content-id=amzn1.sym.6640a844-ab24-4352-ac9b-78899e683a5e&pf_rd_p=6640a844-ab24-4352-ac9b-78899e683a5e&pf_rd_r=SZAWKWXH9GKEB7FK79DX&pd_rd_wg=zuRSF&pd_rd_r=55c3032c-b91e-47b5-85ec-fe5368334871&pd_rd_i=B0DRW6CYGQ&psc=1",
  "seller_id": "Amazon.com",
  "started_at": "2026-10-07T03:58:25.759851+00:00"
}

================== Orchestrator plan ==================
{
  "selected_tools": [
    "collect_product_page",
    "collect_reviews",
    "collect_seller_information"
  ],
  "note": "The investigation still runs required collection/analysis stages even if planning is unavailable."
}

================== Product page collection ==================
{
  "status": "ok",
  "method": 

In [ ]:


# PRODUCT 3

result = investigate(
    product_url="https://www.amazon.com/dp/B09XS7JWHH",
    seller_id="Amazon.com"
)

print(json.dumps(result["report"], indent=2))


PS-13 investigation starting. Collection and analyses run sequentially; unavailable data is reported, not inferred.

================== Input ==================
{
  "product_url": "https://www.amazon.com/dp/B09XS7JWHH",
  "seller_id": "Amazon.com",
  "started_at": "2026-10-06T18:47:27.601446+00:00"
}

================== Orchestrator plan ==================
{
  "selected_tools": [
    "collect_product_page",
    "collect_reviews",
    "collect_seller_information",
    "find_related_listings"
  ],
  "note": "The investigation still runs required collection/analysis stages even if planning is unavailable."
}

================== Product page collection ==================
{
  "status": "ok",
  "method": "firecrawl",
  "source_url": "https://www.amazon.com/dp/B09XS7JWHH",
  "fallback_note": null,
  "message": null,
  "product_snapshot": {
    "title": "Unknown product",
    "url": "https://www.amazon.com/dp/B09XS7JWHH",
    "status": "ok",
    "extracted_at": "2026-10-06T18:47:32.077839+00:0

In [ ]:


# PRODUCT 4

result = investigate(
    product_url="https://www.amazon.com/dp/B0CHWRXH8B",
    seller_id="Amazon.com"
)

print(json.dumps(result["report"], indent=2))


PS-13 investigation starting. Collection and analyses run sequentially; unavailable data is reported, not inferred.

================== Input ==================
{
  "product_url": "https://www.amazon.com/dp/B0CHWRXH8B",
  "seller_id": "Amazon.com",
  "started_at": "2026-10-06T18:50:33.146526+00:00"
}

================== Orchestrator plan ==================
{
  "selected_tools": [
    "collect_product_page",
    "collect_reviews"
  ],
  "note": "The investigation still runs required collection/analysis stages even if planning is unavailable."
}

================== Product page collection ==================
{
  "status": "ok",
  "method": "firecrawl",
  "source_url": "https://www.amazon.com/dp/B0CHWRXH8B",
  "fallback_note": null,
  "message": null,
  "product_snapshot": {
    "title": "Unknown product",
    "url": "https://www.amazon.com/dp/B0CHWRXH8B",
    "status": "ok",
    "extracted_at": "2026-10-06T18:50:35.292364+00:00"
  },
  "content_available": true
}

================== Revie